# Matcher -> OpenVINO IR export + `MatcherOpenVINO` inference

Fits a PyTorch `Matcher` on a reference, bakes the reference features and
post-processing into a single OpenVINO IR on disk, then reloads it with
`MatcherOpenVINO` and runs inference on CPU.

The exported directory contains:

* `matcher.xml` / `matcher.bin` - the baked graph: `target_image -> masks/scores/labels`
* `metadata.json` - input size / patch size / category id->name map used by `MatcherOpenVINO`

Because the reference features are baked into the graph at export time, the
loaded `MatcherOpenVINO` does **not** support `fit()`. To change the reference,
re-run `Matcher.fit(...)` then `Matcher.to_openvino(...)`.

`predict()` takes `Sample` objects and returns `list[Prediction]`.

In [ ]:
from pathlib import Path

from instantlearn.models import Matcher, MatcherOpenVINO
from instantlearn.models.torch_base import ExportConfig
from instantlearn.data.base.sample import Category, Sample
from instantlearn.utils.constants import CompressionMode

EXPORT_DIR = Path("./matcher-openvino")
REF_IMAGE = "assets/coco/000000286874.jpg"
REF_MASK = "assets/coco/000000286874_mask.png"
TARGET_IMAGE = "assets/coco/000000390341.jpg"

## 1. Fit the reference and export the baked IR (INT8 by default)

`to_openvino()` requires `fit()` first - the reference features are baked into
the exported graph as constants. INT4 compression is rejected for Matcher (it
produces noisy masks); use INT8 or no compression.

In [ ]:
matcher = Matcher(device="cpu")
matcher.fit(Sample(image_path=REF_IMAGE, mask_paths=[REF_MASK], categories=[Category(0, "elephant")]))

ir_dir = matcher.to_openvino(
    EXPORT_DIR,
    config=ExportConfig(compression=CompressionMode.INT8_SYM),
)
for path in sorted(ir_dir.iterdir()):
    print(path.name)

## 2. Reload the baked IR with `MatcherOpenVINO` and run inference

`MatcherOpenVINO(model_dir=...)` loads `matcher.xml` + `metadata.json`. The
reference is already baked in, so no `fit()` is needed - just `predict()`.

In [ ]:
ov_model = MatcherOpenVINO(model_dir=EXPORT_DIR, device="CPU")
pred = ov_model.predict(Sample(image_path=TARGET_IMAGE))[0]
print("masks :", pred.masks.shape)
print("scores:", pred.scores)
print("labels:", pred.label_names)

## 3. Visualize

In [ ]:
import cv2
import matplotlib.pyplot as plt
from instantlearn.visualizer import render_predictions, setup_colors

target_rgb = cv2.cvtColor(cv2.imread(TARGET_IMAGE), cv2.COLOR_BGR2RGB)
# render_predictions accepts a Prediction directly.
vis = render_predictions(target_rgb, pred, setup_colors({0: "elephant"}))
plt.figure(figsize=(10, 8))
plt.imshow(vis)
plt.axis("off")
plt.show()